In [1]:
!pip install biopython
!pip install import-ipynb
import import_ipynb
import pandas as pd
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import os
import sys
import shutil
import re

#Mount drive
from google.colab import drive
drive.mount('/content/drive')
%ls
%cd content
%cd drive
%cd MyDrive
%run GeneClassesCloud.ipynb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.3 MB/s eta 0:00:00
Mounted at /content/drive
drive/  sample_data/
[Errno 2] No such file or directory: 'content'
/content
/content/drive
/content/drive/MyDrive


In [2]:
!curl -o datasets 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets'
!chmod +x datasets
!./datasets

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 18.2M  100 18.2M    0     0  9457k      0  0:00:01  0:00:01 --:--:-- 9457k
datasets is a command-line tool that is used to query and download biological sequence data
across all domains of life from NCBI databases.

Refer to NCBI's [download and install](https://www.ncbi.nlm.nih.gov/datasets/docs/v2/download-and-install/) documentation for information about getting started with the command-line tools.

Usage
  datasets [command]

Data Retrieval Commands
  summary     Print a data report containing gene, genome, taxonomy or virus metadata
  download    Download a gene, genome or virus dataset as a zip file
  rehydrate   Rehydrate a downloaded, dehydrated dataset

Miscellaneous Commands
  completion  Generate autocompletion scripts

Flags
      --api-key string   Specify an NCBI API key
      --debug            Emit debugging in

In [ ]:
org = "Homo sapiens"


#Check sys rgv for given orgnism
#if len(sys.argv) > 1:
#  #Find org flag
#  print("Pulling genome for sys.argv[1], ", str(sys.argv[2]))
#  org = sys.argv[1]
#else:
#  print(f"Pulling genome for Homo sapiens")

# Using download command as an alternative to summary
!./datasets download gene taxon 9606 --include product-report --filename homo_sapiens_genes.zip
#%ls

# Unzip the downloaded file
!unzip -o homo_sapiens_genes.zip -d homo_sapiens_genes
%ls homo_sapiens_genes/ncbi_dataset/data/

# Read the data report into a pandas DataFrame
data_report_path = os.path.join("homo_sapiens_genes", "ncbi_dataset", "data", "product_report.jsonl")

if os.path.exists(data_report_path):
    with open(data_report_path, 'r') as f:
        lines = f.read().splitlines()
    df_inter = pd.DataFrame(lines)
    df_inter.columns = ['json_element']
    # Use a list comprehension with json.loads for efficiency and to handle potential errors
    import json # Import json library
    data = [json.loads(line) for line in df_inter['json_element']]
    gene_sum = pd.json_normalize(data)
    display(gene_sum.head())
else:
    print(f"Error: {data_report_path} not found.")

gene_ids = gene_sum['geneId'].tolist()
#print(gene_ids)
print(gene_ids)

In [ ]:
!./datasets download genome taxon "{org}" --include protein
#protein,cds,genome,rna,seq-report

Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    847B 660kB/s
Downloading: ncbi_dataset.zip    8

In [ ]:
%ls
#Unzip ncbi_dataset.zip
!unzip ncbi_dataset.zip
!rm ncbi_dataset.zip
%ls
print("_______________")
%ls ncbi_dataset/data
directories = [f for f in os.listdir("ncbi_dataset/data") if f not in ['assembly_data_report.jsonl','dataset_catalog.json']]
print(directories)
for d in directories:
  fs = os.listdir(os.path.join("ncbi_dataset", "data", d))
  for f in fs:
    print(f)

In [ ]:
def check_cds_against_aaseq(cds, atgs, aa_seq):
    '''Checks that the codons after a given ATG do indeed encode the associated protein. Also returns the codon list'''
    aaCodonVecs = {'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
                   'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
                   'C': ['TGT', 'TGC'],
                   'W': ['TGG'],
                   'E': ['GAA', 'GAG'],
                   'D': ['GAT', 'GAC'],
                   'P': ['CCT', 'CCC', 'CCA', 'CCG'],
                   'V': ['GTT', 'GTC', 'GTA', 'GTG'],
                   'N': ['AAT', 'AAC'],
                   'M': ['ATG'],
                   'K': ['AAA', 'AAG'],
                   'Y': ['TAT', 'TAC'],
                   'I': ['ATT', 'ATC', 'ATA'],
                   'Q': ['CAA', 'CAG'],
                   'F': ['TTT', 'TTC'],
                   'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
                   'T': ['ACT', 'ACC', 'ACA', 'ACG'],
                   '*': ['TAA', 'TAG', 'TGA'],
                   'A': ['GCT', 'GCC', 'GCA', 'GCG'],
                   'G': ['GGT', 'GGC', 'GGA', 'GGG'],
                   'H': ['CAT', 'CAC']}
    for atg in atgs:
      codvec = []
      tf = False
      #Verify that the first codon is indeed ATG
      assert cds[atg:atg+3] == 'ATG'
      #Now loop through the aaseq
      for i in range(len(aa_seq)):
        cod = cds[atg+3*i:atg+3*i+3]
        codvec.append(cod)
        for aa, cods in aaCodonVecs.items():
          if cod in cods:
            aa_from_cds = aa
            break
        aa_from_seq = aa_seq[i]
        if aa_from_cds != aa_from_seq:
          continue
        if i == len(aa_seq) - 1:
          tf = True
          break
      if tf:
        return atg, codvec


def determineProtWeight(aaSeq: str):
    aaSeq = aaSeq.replace("X", "A")
    prot = ProteinAnalysis(aaSeq)
    return prot.molecular_weight()

def clearDownloadDirectory(dest: str):
  if dest in os.getcwd():
    for f in os.listdir():
      if os.path.isfile(f):
        os.remove(f)
      else:
        shutil.rmtree(f)
      #os.remove(os.path.join(dest, f))
  else:
    print("What are tryna delete?")


def saveNaturalGeneObj(obj: NaturalGene):

    print("CWD: ", str(os.getcwd()))
    while "MyDrive" in os.getcwd():
      %cd ..
    %cd MyDrive
    %cd RefGenes
    %cd NHGeneBodySupp

    name = str(obj.geneID) + r'.json'
    #root = r'RefGenes/NHGenes'
    #fname = os.path.join(root, name)
    fname = name
    #print(fname)
    #print("Beginning json dumps of natural gene")
    #for iso in obj.isoforms:
      #assocProtJSON = dataclasses.asdict(iso.associatedProtein)
      #for exon in iso.geneBody:
      #  exonJSON = dataclasses.asdict(exon)
      #print("Can we do it?")
      #isoAsDict = dataclasses.asdict(iso)
      #print(isoAsDict)
      #isoAsDict['associatedProtein'] = assocProtJSON
      #isoAsDict['geneBody'] = exonJSON


    #raise NotImplementedError

    b = json.dumps(dataclasses.asdict(obj))
    print(b)
    with open(fname, 'wb') as infile:
        infile.write(b.encode('utf-8'))
        #Findme
    infile.close()
    print("# Closed")
    %cd ..
    %cd ..
    %cd WorkingFolders
    %cd NCBIDownload

def getSeqFrom(genomicAccessionVersion, genomicRange, sequenceName, rnas, genomes, exonBeginEndOrder):

  for genome in genomes:
    #print(genome)
    if genomicAccessionVersion == genome.id.split(":")[0]:
      #print("FOUND ONE!")
      range_prelim = genome.id.split(":")[1]
      range_prelim = range_prelim.replace("c", "")
      range_prelim = range_prelim.split("-")
      r1 = int(range_prelim[0])
      r2 = int(range_prelim[1])
      #print(r1)
      #print(r2)
      #print("UUUUUU")

      exonsFull = []
      for ex in exonBeginEndOrder:
        rr1 = int(ex['begin'])
        rr2 = int(ex['end'])
        ord = ex['order']

        #print('rr1: ', rr1)
        #print('rr2: ', rr2)
        #print('--')

        if rr1 > rr2:
          rnaDirection = 'minus'
        else:
          rnaDirection = 'plus'
        #print(rnaDirection)

        #This condition works
        if genomicRange['orientation'] == 'plus' and rnaDirection == 'plus':
          scaledBegin = int(rr1) - r1
          scaledEnd = int(rr2) - r1
          q = genome.seq[scaledBegin:scaledEnd]
          #print(q)
          exonsFull.append(q)

        elif genomicRange['orientation'] == 'plus' and rnaDirection == 'minus':
          print("(Case B:)")
          assert abs(r2 - r1) == abs(rr2 - rr1)
          rnaSnipLen = abs(rr2 - rr1)
          scaledBegin = r1 #- int(rr1)
          scaledEnd = r1 + rnaSnipLen #- int(rr2)
          #scaledBegin = r2 - int(rr1)
          #scaledEnd = int(rr2) - r1
          #scaledEnd = r2 - int(rr2)
          #scaledBegin = int(rr2) - r1
          q = genome.seq[scaledBegin:scaledEnd].reverse_complement()
          #w = genome.seq[scaledBegin:scaledEnd]
          #print(genomicAccessionVersion)
          #print(sequenceName)
          #print(scaledBegin)
          #print(scaledEnd)
          #print(q)
          #print(w)
          raise NotImplementedError
          exonsFull.append(q)

        #this condition works
        elif genomicRange['orientation'] == 'minus' and rnaDirection == 'plus':
          #print("(Case C:)")
          scaledBegin =  r1 - int(rr2)
          scaledEnd = r1 - int(rr1) + 1
          #print("(,", scaledBegin, ", ", scaledEnd,  ")")
          q = genome.seq[scaledBegin:scaledEnd]
          #print(q)
          exonsFull.append(q)

        elif genomicRange['orientation'] == 'minus' and rnaDirection == 'minus':
          print("(Case D:)")
          raise NotImplementedError
          scaledBegin = int(rr1) - r1
          scaledEnd = int(rr2) - r1
          q = genome.seq[scaledBegin:scaledEnd]
          print(q)
          exonsFull.append(q)


  for rna in rnas:
    if genomicAccessionVersion == rna.id:
      print("FOUND ONE! ***")

  return exonsFull

def printNatGene(ng: NaturalGene):
  print(type(ng.geneID))
  print(type(ng.geneName))
  print(type(ng.organism))
  print(type(ng.DNASequence))
  print(type(ng.chromosome))
  print(type(ng.spliceAIDonor))
  print(type(ng.spliceAIReceptor))
  print("%%%")
  for iso in ng.isoforms:
    print(type(iso.isoformNumber))
    print(type(iso.associatedProtein))
    print(type(iso.fullSequence))
    print(type(iso.codingSeq))
    print(type(iso.relativeAbundance))
    print(type(iso.geneBody))
    for ex in iso.geneBody:
      print("LLLLLLLLLLLL")
      print(type(ex['seq']))
      print(type(ex['distFromStart']))
      print(type(ex['distFromEnd']))
    print("$$$")

"""
      @dataclass
      class NaturalGene:
        isoforms: list
        geneID: int
        geneName: str
        organism: str
        DNASequence: str
        spliceAIDonor: list
        spliceAIReceptor: list
        chromosome: int
"""

def exonLocations(geneObj : NaturalGene):
  '''Adds the beginning and end numbers to the NaturalGene's exons in the isoform's gene body'''
  for iso in geneObj.isoforms:
    print(iso.isoformNumber)
    for exon in iso.geneBody:
      #get left and right values of the exon in the DNASeq by doing an alignment
      print(exon)
      alignments = pairwise2.align.localms(geneObj.DNASequence, exon['seq'], 2, 1, -1, 0)
      #sort alignments by score
      alignments.sort(key=lambda x: x.score)
      alignment = alignments[0]
      print("Len DNA Seq")
      print(len(geneObj.DNASequence))
      print("Start" , alignment[3])
      print("End", alignment[4])
      print("Length of exon: ", len(exon['seq']))
      print("Diff in start and end: ", abs(alignment[3]-alignment[4]) )
      print("&&&&&&")
      #raise NotImplementedError
      exon['start'] = alignment[3]
      exon['end'] = alignment[4]
  return geneObj


def locate_codons(codvec, codingSeq, atg_start, dna_seq, exsupp):
  clocs = []
  eseq = ''
  for e in exsupp:
    s = e['seq']
    eseq += s
  assert eseq == codingSeq

  r = atg_start
  cnum = 0
  carry = ''
  for e in exsupp:
    elen = len(e['seq'])
    if r > len(e['seq']):
      r -= len(e['seq'])
      continue

    ## pick up remainder of the codon from carry
    if carry != '':
      r = 3 - len(carry)
      cod = carry + e['seq'][:r]
      remseq = e['seq'][r:]
      carry = ''
      clocs.append((cod, 'S'))
      if cod == 'TAA' or cod == 'TAG' or cod == 'TGA':
        break
      cnum += 1
      r = 0


    remseq = e['seq'][r:]
    r = 0
    #Add five prime exon end codons if the exon is under 15 longer than remseq
    if elen - len(remseq) < 15:
      while elen - len(remseq) < 15:
        clocs.append(remseq[:3], 'F')
        remseq = remseq[3:]
        cnum += 1
    #Add internal codons if the remaining seq is over 15 long
    if len(remseq) > 15:
      while len(remseq) > 15:
        clocs.append(remseq[:3], 'I')
        remseq = remseq[3:]
        cnum += 1
    #Add three prime codons if the remseq is under 15 long
    if len(remseq) <= 15:
      while len(remseq) != 0:
        if len(remseq) < 3:
          carry += remseq
          remseq = ''
          continue
        else:
          clocs.append(remseq[:3], 'T')
          if len(remseq) == 3 :
            remseq = ''
          else:
            remseq = remseq[3:]
          cnum += 1
    print(clocs)
    return clocs

def downloadGenePackagesAndProcess(gene_ids, data_directory):
    '''Download a gene package and put it into a NaturalGene object. Save to RefGenes/NHGenes'''
    #Uses a method found here: https://sundararamanp.medium.com/a-relatively-faster-approach-for-reading-json-lines-file-into-pandas-dataframe-90b57353fd38
    #print(os.getcwd())

    for gene_id in gene_ids:
      %ls
      #Fix this to suppress errors
      if "NCBIDownload" not in os.getcwd():
        %cd $data_directory
        %cd NCBIDownload
      fnamezip = str(gene_id) + r'.zip'
      fname = str(gene_id)
      #print(fnamezip)
      #print("RRR")
      #print(fname)
      argg = r'gene,rna,protein,cds,3p-utr,5p-utr,product-report'
      #arrg = r'product-report'
      !datasets download gene gene-id $gene_id --filename $fnamezip --include $argg --no-progressbar
      sleep(5)
      !unzip -o $fnamezip -d $fname
      #cd to the gene data package
      #data_path = os.path.join(data_directory, fname, 'ncbi_dataset', 'data') # Construct the correct path
      # %cd $data_path # No need to change directory, use the full path instead

      # Check if the file exists before attempting to read
      rna_path = os.path.join(fname, 'ncbi_dataset', 'data', 'rna.fna')
      data_path = os.path.join(fname, 'ncbi_dataset', 'data', 'data_report.jsonl')
      pro_path = os.path.join(fname, 'ncbi_dataset', 'data', 'protein.faa')
      futr_path = os.path.join(fname, 'ncbi_dataset', 'data', '5p_utr.fna')
      tutr_path = os.path.join(fname, 'ncbi_dataset', 'data', '3p_utr.fna')
      cds_path = os.path.join(fname, 'ncbi_dataset', 'data', 'cds.fna')
      gene_path = os.path.join(fname, 'ncbi_dataset', 'data', 'gene.fna')
      prodrep_path = os.path.join(fname, 'ncbi_dataset', 'data', 'product_report.jsonl')

      #print("Checking RNA")
      if os.path.exists(rna_path):
          # Use the full path to the RNA file
          rnas = []
          for record in SeqIO.parse(rna_path, 'fasta'):
              #print(record.seq)
              #print(record.id)
              #print(record.description)
              rnas.append(record)
      else:
          print("Error: 'rna.fna' not found in the directory.")

      #print(r"Checking 5'UTR")
      if os.path.exists(futr_path):
          # Use the full path to the RNA file
          futrs = []
          for record in SeqIO.parse(futr_path, 'fasta'):
              #print(record.seq)
              futrs.append(record)
      else:
          print("Error: '5p_utr.fna' not found in the directory.")

      #print(r"Checking cds")
      if os.path.exists(cds_path):
          # Use the full path to the RNA file
          cdss = []
          for record in SeqIO.parse(cds_path, 'fasta'):
              #print(record.seq)
              cdss.append(record)
      else:
          print("Error: 'cds.fna' not found in the directory.")

      #print(r"Checking 3'UTR")
      if os.path.exists(tutr_path):
          # Use the full path to the RNA file
          tutrs = []
          for record in SeqIO.parse(tutr_path, 'fasta'):
              #print(record.seq)
              tutrs.append(record)
      else:
          print("Error: '3p_utr.fna' not found in the directory.")

      #print("Checking proteins")
      if os.path.exists(pro_path):
          # Use the full path to the protein file
          proteins = []
          for record in SeqIO.parse(pro_path, 'fasta'):
              #print(record.seq)
              proteins.append(record)
      else:
          print("Error: 'protein.faa' not found in the directory.")

      #print("Checking genome")
      if os.path.exists(gene_path):
          # Use the full path to the protein file
          genomes = []
          for record in SeqIO.parse(gene_path, 'fasta'):
              #print(record.seq)
              genomes.append(record)
      else:
          print("Error: 'gene.fna' not found in the directory.")

      #print("Checking JSON")
      if os.path.exists(data_path):
          #print("Found JSONL")
          # Use the full path to the JSON file
          with open(data_path) as f:
            lines = f.read().splitlines()
          df_inter = pd.DataFrame(lines)
          df_inter.columns = ['json_element']
          df_inter['json_element'].apply(json.loads)
          df_final = pd.json_normalize(df_inter['json_element'].apply(json.loads))
          #display(df_final)
      else:
          print("Error: 'data_report.jsonl' not found in the directory.")


      #print("Checking product report")
      if os.path.exists(prodrep_path):
          #print("Found product report")
          with open(prodrep_path) as f:
            lines = f.read().splitlines()
          df_inter1 = pd.DataFrame(lines)
          df_inter1.columns = ['json_element']
          df_inter1['json_element'].apply(json.loads)
          df_final1 = pd.json_normalize(df_inter1['json_element'].apply(json.loads))
          #display(df_final1)
      else:
          print("Error: 'product_report.jsonl' not found in the directory.")

      print("Beginning processing for gene " + str(gene_id))

      gSeq = str(genomes[0].seq)
      geneName = df_final['symbol'][0]
      desc = df_final['description'][0]
      organism = df_final['commonName'][0]
      chromosome = df_final['chromosomes'][0]

      if 'synonyms' in df_final:
        synonyms = df_final['synonyms'][0]
      else:
        synonyms = None
      try:
        isoforms = df_final1['transcripts'][0]
      except:
        continue
      try:
        orient = df_final['orientation'][0]
      except:
        display(df_final)
        raise NotImplementedError
      isoSet = []
      exonSets = []
      for iso in isoforms:
        #print("iso")
        #print(iso)


        if 'name' in iso:
          isoformNumber = iso['name'].replace('transcript variant ', '')
        else:
          isoformNumber = -1

        #Transcripts labeled with an X are computationally predicted but not experimentally validated
        try:
          if "X" in isoformNumber:
            continue
        except TypeError:
            pass

        if len(isoforms) == 1:
          isoformNumber = 1
        #print("ISOFORM: ", isoformNumber)

        if 'protein' in iso:
          tmp = iso['protein']
          protFile = tmp['accessionVersion']

          for p in proteins:
            if protFile in p.id:
              pseq = p.seq
              pweight = determineProtWeight(str(pseq))
              associatedProtein = ProteinObj(str(pseq), pweight, gene_id)
        else:
          associatedProtein = ProteinObj(str(""), 0.0, gene_id)

          #associatedProtein = iso['protein']
        geneAccession = iso['accessionVersion']
        if 'cds' in iso:
          cdsAccession = iso['cds']['accessionVersion']
          cds_temp = iso['cds']
          rng_temp = cds_temp['range']
          rng_temp = rng_temp[0]
          cdsRange = [rng_temp['begin'], rng_temp['end']]
        else:
          #print("TTTTTTTTTTTTTTTTTTTTT")
          cdsRange = [-1,-1]
        if 'ensemblTranscript' in iso:
          ensemblTranscript = iso['ensemblTranscript']
        geneLocs = iso['genomicLocations']

        #print("There are ", str(len(geneLocs)), " genomic location sets in ", geneName, " : isoform ", isoformNumber)
        for gLocSet in geneLocs:
          if 'exons' not in gLocSet:
            continue
          #print("LOC!!!")
          #print(gLocSet)
          exonBeginEndOrder = gLocSet['exons']
          genomicAccessionVersion = gLocSet['genomicAccessionVersion']
          genomicRange = gLocSet['genomicRange']
          sequenceName = gLocSet['sequenceName']
          #print("** ** ", genomicAccessionVersion)
          #print("SQ!!!")
          exons = getSeqFrom(genomicAccessionVersion, genomicRange, sequenceName, rnas, genomes, exonBeginEndOrder)
          exSupp = []
          for ex in exons:
            #print(ex)
            distFromStart = list(range(0, len(ex)))
            exSupp.append({"seq": str(ex), "distFromStart": str(distFromStart)})
          exonSets.append(exSupp)
        cdsSet = []
        for exSet in exonSets:
          cds = ''
          for ex in exSet:
            cds += ex['seq']
          cdsSet.append(cds)

        assert len(cdsSet) == len(exonSets)



        for codingSeq in cdsSet:
          relativeAbundance = -1
          if associatedProteinObj.aaSeq != '':
            #Find 'ATG's in codingSeq
            atg_indexes = [match.start() for match in re.finditer('ATG', codingSeq)]
            atg_start, codvec = check_cds_against_protein(codingSeq, atg_indexes, associatedProteinObj.aaSeq)
            creg = codingSeq[atg_start:atg_start+3*len(associatedProteinObj.aaSeq)]
            codlocvec = locate_codons(codvec, codingSeq, atg_start, gSeq, exSupp)
            #Write locate_codons!!!

                                                            #Used to be gSeq
                                                                  #
          igb = IsoformGeneBody(isoformNumber, associatedProtein, codingSeq, codvec, relativeAbundance, exSupp)
          isoSet.append(igb)

        #print("GGGGGGGGGGG", len(isoSet))
        refinedIsos = []
        for iso in isoSet:
          if iso.geneBody != []:
            #Eliminate non-unique isoforms
            if iso not in refinedIsos:
              refinedIsos.append(iso)
        isoSet = refinedIsos
        #print("UUUUUUUUUU", len(isoSet))
        refinedIsos = []
        protSeqs = []
        for iso in isoSet:
          if iso.geneBody != []:
            #Eliminate non-unique isoforms
            if str(iso.associatedProtein.aaSeq) not in protSeqs:
              protSeqs.append(str(iso.associatedProtein.aaSeq))
              refinedIsos.append(iso)
        isoSet = refinedIsos
        #print("MMMMMMMMMMMMM", len(isoSet))
        refinedIsos = []
        exonSeqs = []
        for iso in isoSet:
          if iso.geneBody != []:
            #Eliminate non-unique isoforms
            linked = ''
            for exon in iso.geneBody:
              linked += exon['seq']
            if linked not in exonSeqs:
              exonSeqs.append(linked)
              refinedIsos.append(iso)
            #isoSet = refinedIsos
        #print("XXXXXXXXXXXXX", len(isoSet))
        #print("XXXXXXXXXXXXX", len(refinedIsos))
      print("!@#$%")
      ng = NaturalGene(isoSet, gene_id, geneName, organism, gSeq, [], [], {}, chromosome)
      print("Exonizing")
      #ng = exonLocations(ng)
      #printNatGene(ng)
      print("Exonized")


      #raise NotImplementedError
      saveNaturalGeneObj(ng)
      #Clear NCBIDownloads
      clearDownloadDirectory(r'NCBIDownload')
      print("Done with gene " + str(gene_id))

In [ ]:
download

In [ ]:
interfiles = []
for f in fyles:
  print(f)
  p = os.path.join(os.getcwd(),"ncbi_dataset", "data", f, 'protein.faa')
  #load proteins from the file
  for seq_record in SeqIO.parse(p, 'fasta'):
    print(seq_record.id)
    print(repr(seq_record.seq))
    print(seq_record.description)
    print(seq_record.name)
    print("+++++++++++++++++++++++")
  print("OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO")
